# Lecture 6 (Dictionaries and sets)

## Exercise 6.1 (histogram)

Write a method histogram that given a list of values, returns a list of pairs (value, frequency).

_Example_. `histogram(['A', 'B', 'A', 'A', 'C', 'E', 'C'])` should return `[('A', 3), ('B', 1), ('C', 2), ('E', 1)]`.

_Hint_. Use a dictionary and the dictionary method `get`.

_Note_. In the standard library module `collections` the `Counter` method implements the same functionality.

    import collections
    histogram = list(collections.Counter(['A', 'B', 'A', 'A', 'C', 'E', 'C']).items())

## Exercise 6.2 (frequent long words)

_This exercise extends Exercise 6.1._

Write a program that prints the most frequent words containing at least six characters occurring in a text file. E.g. for the [saxo.txt](http://www.gutenberg.org/ebooks/1150.txt.utf-8) file it should print something like the below.

    Rank Freq. Word
    ====================
       1   372 should
       2   221 himself
       3   199 father
       4   196 battle
       5   187 though
       6   181 thought
       7   172 against
       8   142 before
       9   134 daughter
      10   124 country

You can use the following code to split the content of a file into a list of lower case words, ignoring everything that is not a letter in the text.

    import re
    txt = open('saxo.txt', encoding='utf8').read()
    txt = txt.lower()
    words = re.split('[^a-z]+', txt)

_Hint_. Use the `sorted` function to sort a list of tuples of the form (frequency, word), and try to use list comprehension to create such a list from a frequency dictionary.

## Exercise 6.3 - handin 3 (triplet distance - part I)

_This handin together with the next handin constitutes one smaller project. The code from the first project will be used in the second project. In this first project the aim should be to write elegant code using Python's tuples and list comprehensions._

     morphological        18S rDNA
    characteristics     sequence data

       /\                 /\
      /  \               /  \
    'A'  /\            'A'  /\
        /  \               /  \           A = Glycine max
      'C'  /\            'G'  /\          B = Characium perforatum
          /  \               /  \         C = Friedmannia isaelensis
         /    \             /    \        D = Parietochloris pseudoalveolaris
        /\     \           /\     \       E = Dunaliella parva
       /  \    /\         /  \    /\      F = Characium hindakii
     'E'  'G' /  \      'E'  'F' /  \     G = Chlamydomonas
             /    \             /    \
            /\    'D'          /\    'D'
           /  \               /  \
         'B'  'F'           'B'  'C'

    Left:  ('A', ('C', (('E', 'G'), (('B', 'F'), 'D'))))
    Right: ('A', ('G', (('E', 'F'), (('B', 'C'), 'D'))))

_Background_. In this and the next handin we will implement an algorithm to compute the so called _triplet distance_ between two rooted binary trees. The notion of triplet distance was introduced in a paper by Dobson in 1975 and was e.g. considered in the context of Phylogenetic Trees by Critchlow et al. in 1996 (the references are provided to show the scientific background of this exercise - it is not necessary to read the papers to solve this exercise).

* Annette J. Dobson,
  **Comparing the Shapes of Trees**.
  Combinatorial Mathematics III, Lecture Notes in Mathematics, volume 452, 95-100, 1975,
  doi: [10.1007/BFb0069548](https://doi.org/10.1007/BFb0069548).

* Douglas E. Critchlow Dennis K. Pearl Chunlin Qian,
  **The Triples Distance for Rooted Bifurcating Phylogenetic Trees**.
  _Systematic Biology_, 45(3):323-334, 1996,
  doi: [10.1093/sysbio/45.3.323](https://doi.org/10.1093/sysbio/45.3.323).

Above is an example from the paper by Critchlow et al. showing two phylogenies for chloroccalean zoosporic green algae, generated based on morphological characteristics (left) and on 18S rDNA sequence data (right).

The _triplet distance_ (defined below) between the two binary trees is a measure on how different the two resulting trees are. There are many definitions of distance measures between trees - we will in this exercise only consider the triplet distance between two rooted binary trees.

_Tree representations_. We restrict the input to our algorithm to be rooted binary trees, i.e. trees where all internal nodes have exactly two children. We assume that a binary tree is represented by a recursive tuple with leaves being strings, representing the _labels_ of the leaves. For a single tree we require all leaf labels to be distinct, and for two trees to be compared that they have exactly the same set of leaf labels. Below two binary trees are shown with the same leaf labels `'A'`-`'F'`.

    ((('A','F'),'B'),('D',('C','E')))   (((('D','A'),'B'),'F'),('C','E'))

               (a)                                     (b)

                /\                                      /\
               /  \                                    /  \
              /    \                                  /    \
             /      \                                /      \
            /        \                              /        \
           /          \                            /          \
          /\          /\                          /\          /\
         /  \        /  \                        /  \        /  \
        /    \      /    \                      /    \     'C'  'E'
       /\    'B'  'D'    /\                    /      \
      /  \              /  \                  /\      'F'
    'A'  'F'          'C'  'E'               /  \
                                            /    \
                                           /\    'B'
                                          /  \
                                        'D'  'A'

For a tree with _n_ labels the number of subsets containing three labels equals binomial(_n_, 3) = _n_ · (_n_ - 1) · (_n_ - 2) / 6.  Each such set of three labels defines a _triplet_ in each input tree, i.e. the three leaves with the three labels induce a subtree with three leaves. Below we show the triplets induced for the three labels `{'A', 'D', 'F'}`.  The `'*'` marks the lowest common ancestor (LCA) of the three labels. We say that these nodes are the _anchors_ of the triplets in the two trees.

               (a)                                    (b)

         anchor *                                      /\
               / \                                    /  \
              /   \                                  /    \
             /     \                                /      \
            /       \                              /        \
           /         \                            /          \
          /\         /\                   anchor *           /\
         /  \       /  \                        / \         /  \
        /    \     /    \                      /   \      'C'  'E'
       /\    'B' 'D'    /\                    /     \
      /  \             /  \                  /\     'F'
    'A'  'F'         'C'  'E'               /  \
                                           /    \
                                          /\    'B'
                                         /  \
                                       'D'  'A'

                  Induced triplets by {'A', 'D', 'F'}

                /\                                    /\
               /  \                                  /  \
              /    \                                /    \
             /\    'D'                             /\    'F'
            /  \                                  /  \
          'A'  'F'                              'D'  'A'

        (('A', 'F'), 'D')                     (('D', 'A'), 'F')

Since we only care about the topologies of the induced trees, and not if a child is the left or right child of its parent, we for each triplet define its unique _canonical triplet representation_. For a triplet anchored at a node, with label _a_ in one subtree and _b_ and _c_ in the other subtree where _b_ ≤ _c_, we define the canonical representation as the triplet where _a_ is in the left subtree and _b_ and _c_ are in the right subtree, with _b_ to the left of _c_. Below are the canonical triplet representations of the two triplets above:

        Induced canonical triplets by {'A', 'D', 'F'}

           /\                                    /\
          /  \                                  /  \
         /    \                                /    \
       'D'    /\                             'F'    /\
             /  \                                  /  \
           'A'  'F'                              'A'  'D'

    ('D', ('A', 'F'))                     ('F', ('A', 'D'))

**Definition**: Given two trees, where each tree has _n_ distinctly labeled leaves and the two trees have identical label sets, the _triplet distance_ between the two trees equals _n_ · (_n_ - 1) · (_n_ - 2) / 6 minus the number of label subsets of size three with identical induced canonical triplet representations in both trees.

_For each of the following questions try to make efficient use of Python's tuples and list comprehension._

1.  Make a function `generate_labels(n)`, that given an integer `n` returns a list of `n` _distinct strings_, e.g. `'A'`, `'B'`, ... or `'L1'`, `'L2'` ...

    _Example_. `generate_labels(5)` could return `['A', 'B', 'C', 'D', 'E']`.


2.  Make a function `permute(L)`, that given a list `L`, returns a new list containing a _random permutation_ of the elements in `L`.

    _Hint_. Construct the new list left-to-right by randomly selecting an element not selected so far.  To generate a random integer in the interval [a, b], you can you the function `randint(a, b)` from the module [`random`](https://docs.python.org/3/library/random.html) (use `from random import randint` to get access to the function).

    _Note_. Using the functions `shuffle` or `sample` from the module `random` to solve the question would be considered cheating.

    _Example_. `permute(['A', 'B', 'C'])` could return `['B', 'C', 'A']`.


3.  Make a function `pairs(L)`, that given a list of comparable elements, returns a list of all pairs, i.e. tuples with two elements, `(a, b)` where `a` < `b`.

    _Example_. `pairs(['A', 'F', 'B'])` should return `[('A', 'F'), ('A', 'B'), ('B', 'F')]`.


4.  Make a function `canonical_triplets(A, B)` that returns a list of all canonical triples where the left subtree contains a label from `A` and the right subtree is a pair from `B`.

    _Example_. `canonical_triplets(['A', 'B'], ['C', 'D', 'E'])` should return: `[('A', ('C', 'D')), ('A', ('C', 'E')), ('A', ('D', 'E')), ('B', ('C', 'D')), ('B', ('C', 'E')), ('B', ('D', 'E'))]`.


5.  Make a function `anchored_triplets(L, R)` that returns a list of all canonical triples anchored at a node _v_ where the leaves in the left subtree of _v_ contains the labels in the list `L` and the leaves in the right subtree of _v_ contains the labels in the list `R`.

    _Example_. For the root of the tree (a) `anchored_triplets(['A', 'F', 'B'], ['D', 'C', 'E'])` should return the following 18 canonical triplets: `[('A', ('D', 'E')), ('A', ('C', 'D')), ('A', ('C', 'E')), ('F', ('D', 'E')), ('F', ('C', 'D')), ('F', ('C', 'E')), ('B', ('D', 'E')), ('B', ('C', 'D')), ('B', ('C', 'E')), ('D', ('A', 'F')), ('D', ('A', 'B')), ('D', ('B', 'F')), ('C', ('A', 'F')), ('C', ('A', 'B')), ('C', ('B', 'F')), ('E', ('A', 'F')), ('E', ('A', 'B')), ('E', ('B', 'F'))]`.


_Handin format_. As in handin 1 one .py file with a docstring with reflection.